# Staggered-grid (WRF) NetCDF files

WRF model output with mass and staggered (`*_stag`) dimensions and string bookkeeping variables. Inspection, plotting, retrieval, and variable mutation; affine polygon crop does not apply.

In [ ]:
%matplotlib inline
from pathlib import Path
import tempfile

import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon

from pyramids.netcdf import NetCDF, UgridDataset
from pyramids.feature import FeatureCollection

DATA = Path('../../../../examples/data/netcdf/samples')

## `none__17v__1d1-2d5-3d6-4d5__stag-str.nc`

WRF surface/3-D fields on mass and staggered grids (T2 = 2 m temperature).

**Open the file and inspect the container**

In [ ]:
nc = NetCDF.read_file(DATA / 'none__17v__1d1-2d5-3d6-4d5__stag-str.nc')
nc

**Dimensions and variables**

In [ ]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

**Global attributes**

In [ ]:
nc.global_attributes

**Plot the variable** (a 2-D slice is auto-selected for >2-D variables)

In [ ]:
glyph = nc.plot(variable='T2')
# overlay the Natural Earth coastline for geographic context — the glyph carries
# the data CRS (issue #630), so no crs= is needed to line it up
glyph.add_features("coastline", "50m", zorder=5)

**Retrieve the underlying data**

In [ ]:
var = nc.get_variable('T2')
data = var.read_array()
print('shape:', data.shape)
print('min / mean / max:', float(np.nanmin(data)), float(np.nanmean(data)), float(np.nanmax(data)))

**Add a variable** — derive a 2-D field and append it as a new variable

In [ ]:
work = Path(tempfile.mkdtemp())
slice2d = data[tuple(0 for _ in range(data.ndim - 2))]
NetCDF.create_from_array(
    arr=slice2d, geo=(0.0, 1.0, 0.0, 0.0, 0.0, -1.0), epsg=var.epsg or 4326,
    variable_name='T2_slice0', path=str(work / 'derived.nc'),
)
nc.add_variable(NetCDF.read_file(str(work / 'derived.nc')), 'T2_slice0')
print('variables after add:', nc.variable_names)

**Remove a variable**

In [ ]:
nc.remove_variable('T2_slice0')
print('variables after remove:', nc.variable_names)

**Crop with a polygon** — WRF output is on a **staggered, map-projected** grid, but its cell centres carry 2-D geolocation (`XLAT(y, x)` / `XLONG(y, x)`). `crop` masks on those 2-D coordinates (the same path as the ROMS curvilinear example), so a polygon crop works and the result stays curvilinear.

In [ ]:
# the WRF domain spans roughly lon -94..-66, lat 25..44; crop to a box over the north-eastern US
aoi = FeatureCollection(gpd.GeoDataFrame(
    geometry=[Polygon([(-88, 35), (-75, 35), (-75, 43), (-88, 43)])], crs=4326))
t2 = nc.get_variable('T2')
cropped = t2.crop(aoi)
print('cropped shape:', cropped.shape)
glyph = cropped.plot()
# overlay the coastline so the cropped WRF patch is easy to place
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Save the container to a new NetCDF file**

In [ ]:
out = work / 'saved.nc'
nc.to_file(out)
print('saved container to', out.name, '->', out.exists())